# W02b.1 — LLM API Basics with LangChain

We'll start with the basics of LLM API calls with LangChain.

The rest of the notebook assumes you've installed the required packages and set up your .env file with the required keys as shown in class.

In [ ]:
# import the API keys
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 1. Your first call

LangChain provides a small number of core features to get started with AI agents: `chat_models`, `messages`, `tools`, `agents`.

`init_chat_model` is the universal constructor for a chat model: give it a model name, get back a model with the same interface regardless of provider.

### Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
claude = init_chat_model(model="claude-sonnet-4-6")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

### Sending an inference request

Use `invoke` to send a request to the model.

In [ ]:
question = "In one sentence: What does temperature do in an API call?"

In [ ]:
#TODO: send the request and store the response
#TODO: display the response

`invoke` returned an `AIMessage` object, not a string. It carries the text plus metadata about how it was produced. Look at what else is in there.

In [ ]:
from pprint import pprint

print(type(response).__name__)
pprint(response.response_metadata)

## 2. Messages and roles

APIs take a list of messages, each with a role:

| Role | Class | Who it is |
|---|---|---|
| system | `SystemMessage` | You, the developer: persona, rules |
| user | `HumanMessage` | The end user |
| assistant | `AIMessage` | The model's own past turns |

The system prompt is used to steer the assistant persona.

Example: `SystemMessage(content="your message")`

In [ ]:
system_prompt = "You are a teaching assistant for an Agentic AI course. Answer in one sentence."
user_question = "What does temperature do in an API call?"

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

#TODO: define the list of messages containing system and user prompts.

response = model.invoke(messages)
print(response.content)

## 3. The model is stateless

The model keeps no memory between calls. If you want a conversation, you must send the whole history every time. 

Let's see this in action. First turn:

In [ ]:
from langchain.messages import AIMessage

conversation = [
    SystemMessage(content="You are a concise tutor."),
    HumanMessage(content="Hello from Boston. Give me a two-line analogy for what an embedding is."),
]

first_response = model.invoke(conversation)


In [ ]:
type(first_response)

In [ ]:
first_response

In [ ]:
next_message = HumanMessage(content="Where do I live?")
next_response = model.invoke([next_message])
print(next_response.content)

In [ ]:
print(first_response.content)

To help the model remember previous conversations, we need to append the model's reply to the history:

In [ ]:
pprint(conversation)

In [ ]:
conversation.append(first_response)

In [ ]:
pprint(conversation)

In [ ]:
model.invoke([next_message])

We need to append the follow-up to the original list of messages and send everything again:

In [ ]:
conversation

In [ ]:
conversation.append(next_message)

In [ ]:
second_message = model.invoke(conversation)
print(second_message.content)

In [ ]:
new_message = HumanMessage("Give me another analogy that's more relatable to someone who loves cooking.")
conversation.append(new_message)

In [ ]:
last_response = model.invoke(conversation)

print(last_response.content)

In [ ]:
pprint(conversation)

Every extra turn grows the context the model re-reads. This is why long agent sessions get expensive.

## 4. The sampling knobs

As we discussed, the model outputs a probability distribution over the next token. The API knobs decide how we draw from it.

- `temperature` divides the logits before softmax. Low sharpens (more deterministic), high flattens.
- `top_p` keeps only the smallest set of tokens covering probability mass p.
- `max_tokens` is a hard cap on output length.

BUT: reasoning-first models (the `gpt-5` family) fix their sampling internally. For these experiments we'll use a standard chat model without reasoning.

In [ ]:
#sampler = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
sampler = ChatOllama(model="qwen3.5:4b", reasoning=False)
#sampler = init_chat_model(model="gpt-4.1-mini")


Sampling is random, so a single call proves nothing. A small helper to run the same prompt several times:

In [ ]:
def sample_n(prompt, n=4, **model_kwargs):
    #m = init_chat_model(model="gpt-4.1-mini", **model_kwargs)
    m = ChatOllama(model="qwen3.5:4b", reasoning=False, **model_kwargs)
    return [m.invoke(prompt).content.strip() for _ in range(n)]

prompt = "Invent a name for Northeastern University's mascot, husky. Reply with the name only."


In [ ]:
print("temperature = 0.0  (near-greedy)")

for out in sample_n(prompt, temperature=0.0):
    print(" ", out)

In [ ]:
print("temperature = 1.4  (flattened distribution)")
for out in sample_n(prompt, temperature=1.4):
    print(" ", out)

Near-identical names at 0.0, real variety at 1.4. Two caveats: even at temperature 0 the answers can occasionally differ, and high temperature is not more intelligence, only a flatter distribution.

In [ ]:
print("top_p = 0.1  (only the most probable tokens survive)")
for out in sample_n(prompt, top_p=0.1):
    print(" ", out)

### Choosing settings:

| Task | Setting |
|---|---|
| Tool calls, structured output | temperature 0 to 0.3 |
| Factual Q&A, extraction | low 0.3 to 0.5 |
| Brainstorming, drafts | 0.8 to 1.2, several samples |
| General chat | defaults; tune one knob, not both |

**Rule of thumb for agents: agents loop, and a small weirdness rate per step compounds across twenty steps. Keep agent temperature low.**

### max_tokens
It's also possible to limit the number of tokens to be generated. Use `max_tokens` as a cost ceiling. 

In [ ]:
#capped = ChatOllama(model="qwen3.5:4b", reasoning=False, max_tokens = 50)
capped = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", max_tokens=50)
r = capped.invoke("Explain how backpropagation works.")


In [ ]:
pprint(r.content)
pprint("\nstop reason:", r.response_metadata.get("finish_reason") or r.response_metadata.get("stop_reason"))

The text stops mid-sentence and the stop reason says `length`, not `stop`. 

## 5. Streaming

Generation is one token per forward pass. Streaming exposes that rhythm instead of making you wait for the full reply.

Use `model.stream()` to stream each chunk as you receive them. Will need to iterate through the 
- model.stream("your query") will return the response as a sequence

In [ ]:
#TODO: iterate through the stream
#TODO: display the chunks
